######

# Fase 2 — Data Understanding



# Fase 2 — Data Understanding
## Construcción y validación de DENUE 2020
### Proyecto
Modelo econométrico de atractividad comercial municipal en México.

### Objetivo de esta etapa
Exploración y validación de los archivos DENUE correspondientes al comercio al por menor en 2020 antes de construir la base municipal.

Esta etapa se revisa los siguientes archivos:

- disponibilidad de los archivos;
- estructura interna de los archivos ZIP;
- nombres de los archivos CSV;
- número de registros;
- variables disponibles;
- tipos de datos;
- códigos de actividad económica;
- claves de entidad y municipio;
- valores faltantes;
- duplicados;
- cobertura municipal.



In [1]:
from pathlib import Path
import zipfile

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
# Ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Ruta DENUE 2020
DENUE_2020_DIR = PROJECT_ROOT / "data" / "raw" / "denue" / "2020"

print("Ruta raíz del proyecto:")
print(PROJECT_ROOT)

print("\nRuta DENUE 2020:")
print(DENUE_2020_DIR)

print("\n¿Existe la carpeta?:", DENUE_2020_DIR.exists())

Ruta raíz del proyecto:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico

Ruta DENUE 2020:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\raw\denue\2020

¿Existe la carpeta?: True


In [3]:
archivos_zip = sorted(DENUE_2020_DIR.glob("*.zip"))

print(f"Número de archivos ZIP encontrados: {len(archivos_zip)}\n")

for archivo in archivos_zip:
    print(archivo.name)

Número de archivos ZIP encontrados: 4

denue_00_46111_1120_csv.zip
denue_00_46112-46311_1120_csv.zip
denue_00_46321-46531_1120_csv.zip
denue_00_46591-46911_1120_csv.zip


### 2.1 Inspección de la estructura interna de los archivos DENUE 2020

Antes de cargar los datos se revisa el contenido de cada archivo ZIP para identificar:

- archivo principal de datos;
- diccionario de datos;
- metadatos;
- estructura de carpetas.

Esta inspección permite validar que los cuatro paquetes DENUE presentan una estructura consistente.

In [4]:
for archivo_zip in archivos_zip:

    print("=" * 90)
    print(f"ARCHIVO: {archivo_zip.name}")
    print("=" * 90)

    with zipfile.ZipFile(archivo_zip, "r") as z:
        contenido = z.namelist()

        for elemento in contenido:
            print(elemento)

    print()

ARCHIVO: denue_00_46111_1120_csv.zip
diccionario_de_datos/denue_diccionario_de_datos.csv
conjunto_de_datos/denue_inegi_46111_.csv
metadatos/metadatos_denue.txt

ARCHIVO: denue_00_46112-46311_1120_csv.zip
diccionario_de_datos/denue_diccionario_de_datos.csv
conjunto_de_datos/denue_inegi_46112-46311_.csv
metadatos/metadatos_denue.txt

ARCHIVO: denue_00_46321-46531_1120_csv.zip
diccionario_de_datos/denue_diccionario_de_datos.csv
conjunto_de_datos/denue_inegi_46321-46531_.csv
metadatos/metadatos_denue.txt

ARCHIVO: denue_00_46591-46911_1120_csv.zip
diccionario_de_datos/denue_diccionario_de_datos.csv
conjunto_de_datos/denue_inegi_46591-46911_.csv
metadatos/metadatos_denue.txt



### 2.2 Lectura controlada de los archivos principales

Antes de cargar completamente las bases DENUE 2020, se realiza una lectura de muestra de cada archivo.

El objetivo es verificar:

- nombres de columnas;
- estructura de los registros;
- consistencia entre los cuatro archivos;
- disponibilidad de las claves geográficas;
- disponibilidad del código de actividad económica;
- disponibilidad del identificador de cada establecimiento.

Para evitar un uso innecesario de memoria, en esta etapa se leen únicamente las primeras filas de cada archivo.

In [6]:
# Diccionario para almacenar las muestras
muestras_denue = {}

for archivo_zip in archivos_zip:

    with zipfile.ZipFile(archivo_zip, "r") as z:

        # Identificar automáticamente el CSV principal
        archivos_datos = [
            nombre for nombre in z.namelist()
            if nombre.startswith("conjunto_de_datos/")
            and nombre.endswith(".csv")
        ]

        if len(archivos_datos) != 1:
            print(
                f"Advertencia en {archivo_zip.name}: "
                f"se encontraron {len(archivos_datos)} archivos de datos."
            )
            continue

        csv_principal = archivos_datos[0]

        # Leer únicamente las primeras 5 filas
        with z.open(csv_principal) as archivo_csv:
            muestra = pd.read_csv(
                archivo_csv,
                nrows=5,
                encoding="latin-1"
            )

        muestras_denue[archivo_zip.name] = muestra

        print("=" * 100)
        print(f"ZIP: {archivo_zip.name}")
        print(f"CSV: {csv_principal}")
        print(f"Número de columnas: {muestra.shape[1]}")
        print("=" * 100)

        display(muestra)

ZIP: denue_00_46111_1120_csv.zip
CSV: conjunto_de_datos/denue_inegi_46111_.csv
Número de columnas: 41


,id,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,nom_v_e_1,tipo_v_e_2,nom_v_e_2,tipo_v_e_3,nom_v_e_3,numero_ext,letra_ext,edificio,edificio_e,numero_int,letra_int,tipo_asent,nomb_asent,tipoCenCom,nom_CenCom,num_local,cod_postal,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,45602,AABARROTES LOS PINOS,NaN,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,CALLE,SIERRA DE LA GAVIA,CALLE,SIERRA DE TLAXCO,CALLE,SIERRA DE LA GLORIA,CALLE,SIERRA DEL MAGUEY,221,NaN,NaN,NaN,NaN,NaN,FRACCIONAMIENTO,LAS CUMBRES,NaN,NaN,NaN,20175,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,2352,27,NaN,NaN,NaN,Fijo,21.911178,-102.256462,2014-12
1,35543,ABAROTES MARCOS,NaN,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,CALLE,SARGENTO LIBERATO SANTA CRUZ,CALLE,DOLORES,CALLE,GENERAL ENRIQUE ESTRADA,CALLE,MELQUÍADES MORENO,604,NaN,NaN,NaN,NaN,NaN,COLONIA,GREMIAL,NaN,NaN,NaN,20030,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,052A,17,NaN,NaN,NaN,Fijo,21.893444,-102.291618,2014-12
2,20536,ABAROTES MORALES,NaN,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,CALLE,COSIO SUR,CALLE,JUAN DE MONTORO,CALLE,FRANCISCO G. HORNEDO,CALLE,JOSEFA ORTIZ DE DOMÍNGUEZ,117,NaN,NaN,NaN,0.0,NaN,COLONIA,ZONA CENTRO,NaN,NaN,NaN,20000,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,0712,9,4.499155e+09,NaN,NaN,Fijo,21.881855,-102.286655,2010-07
3,8797198,ABAROTES ROQUE CORDOVA,NaN,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,CALLE,PIÑA,PRIVADA,UVA,CALLE,GUAYABA,CALLE,NOGAL,106,1.0,NaN,NaN,NaN,NaN,LOCALIDAD,CANADA GRANDE DE COTORINA,NaN,NaN,NaN,20394,1,AGUASCALIENTES,1,Aguascalientes,125,Cañada Grande de Cotorina,1763,3,NaN,NaN,NaN,Fijo,21.782423,-102.237041,2019-11
4,8500037,ABAROTES WILLIE,NaN,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,AVENIDA,CONSTITUCION,CALLE,ARTÍCULO CUARTO,CALLE,ARTICULO TERCERO,AVENIDA,CONSTITUCIÓN,314,NaN,NaN,NaN,0.0,NaN,COLONIA,CONSTITUCION,NaN,NaN,NaN,20126,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,3153,21,NaN,NaN,NaN,Fijo,21.918689,-102.286027,2019-11


ZIP: denue_00_46112-46311_1120_csv.zip
CSV: conjunto_de_datos/denue_inegi_46112-46311_.csv
Número de columnas: 41


,id,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,nom_v_e_1,tipo_v_e_2,nom_v_e_2,tipo_v_e_3,nom_v_e_3,numero_ext,letra_ext,edificio,edificio_e,numero_int,letra_int,tipo_asent,nomb_asent,tipoCenCom,nom_CenCom,num_local,cod_postal,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,8608708,502402 AGCS II COBRANZA AGUASCALIENTES II,COPPEL SA DE CV,462210,Comercio al por menor en tiendas departamentales,31 a 50 personas,CALLE,LIBERTAD,CALLE,JESUS MARIA,CALLE,JESUS TERAN,CALLE,VALLADOLID,711,NaN,NaN,NaN,NaN,NaN,COLONIA,DEL CARMEN,NaN,NaN,NaN,20050,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,0500,1,NaN,NaN,NaN,Fijo,21.889409,-102.299286,2019-11
1,6846393,50PJ1 RANCHO SANTA MONICA AGU,CADENA COMERCIAL OXXO SA DE CV,462112,Comercio al por menor en minisupers,6 a 10 personas,AVENIDA,SAN ANTONIO,CALLE,TENOPALA,AVENIDA,SAN ANTONIO,VIADUCTO,NINGUNO,3,NaN,NaN,NaN,NaN,NaN,COLONIA,SANTA MONICA,NaN,NaN,NaN,20342,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,401A,9,NaN,ATENCIONCLIENTES@OXXO.COM,WWW.OXXO.COM,Fijo,21.837315,-102.316270,2018-03
2,9223444,541801 CLVL COBRANZA CALVILLO,COPPEL SA DE CV,462210,Comercio al por menor en tiendas departamentales,11 a 30 personas,CALLE,5 de Mayo,CALLE,Matamoros,CALLE,Vicente Guerrero,CALLE,5 DE MAYO,211,NaN,NaN,NaN,0.0,NaN,COLONIA,CENTRO,NaN,NaN,NaN,20800,1,AGUASCALIENTES,3,Calvillo,1,Calvillo,0187,5,NaN,NaN,NaN,Fijo,21.846346,-102.720346,2020-04
3,6282006,6046 WALDOS AGUASCALIENTES CONVENCION,WALDO S DOLAR MART DE MEXICO S DE RL DE CV,462111,Comercio al por menor en supermercados,11 a 30 personas,AVENIDA,CONVENCION DE 1914 PONIENTE,PROLONGACION,DOCTOR SALVADOR QUEZADA LIMON,CALLE,TLAXCALA,CALLE,26 DE MARZO,1010,NaN,NaN,NaN,NaN,NaN,COLONIA,GOMEZ,NaN,NaN,NaN,20060,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,3295,11,NaN,NaN,WWW.WALDOS.COM,Fijo,21.885399,-102.312649,2010-07
4,6282007,6055 WALDOS AGUASCALIENTES DE LA CRUZ,WALDO S DOLAR MART DE MEXICO S DE RL DE CV,462111,Comercio al por menor en supermercados,11 a 30 personas,BOULEVARD,JOSE MARIA CHAVEZ,CALLE,REPUBLICA DE ECUADOR,CALLE,EPIFANIO DE SILVA,CALLE,CORONEL VALENTE ARTEAGA,801,NaN,NaN,NaN,NaN,NaN,COLONIA,OBRAJE,NaN,NaN,NaN,20230,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,0854,6,NaN,NaN,WWW.WALDOS.COM,Fijo,21.870941,-102.295287,2010-07


ZIP: denue_00_46321-46531_1120_csv.zip
CSV: conjunto_de_datos/denue_inegi_46321-46531_.csv
Número de columnas: 41


,id,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,nom_v_e_1,tipo_v_e_2,nom_v_e_2,tipo_v_e_3,nom_v_e_3,numero_ext,letra_ext,edificio,edificio_e,numero_int,letra_int,tipo_asent,nomb_asent,tipoCenCom,nom_CenCom,num_local,cod_postal,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,8541319,+ VISION,GMVM SA DE CV,464121,Comercio al por menor de lentes,0 a 5 personas,AVENIDA,CONVENCION DE 1914 ORIENTE,AVENIDA,JOSE H. ESCOBEDO,AVENIDA,SIGLO XIX,CALLE,CANAL INTERCEPTOR,904.0,L06,NaN,NaN,NaN,NaN,FRACCIONAMIENTO,LOMAS DE SANTANITA,NaN,NaN,NaN,20169,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,1354,15,NaN,MV12933@OPTICAS.MASVISION.MX,NaN,Fijo,21.890777,-102.274040,2019-11
1,9287941,+ VISION,GVMV SA DE CV,464121,Comercio al por menor de lentes,0 a 5 personas,CALLE,MONTES HIMALAYA,BOULEVARD,BOULEVARD A ZACATECAS,AVENIDA,INDEPENDENCIA,BOULEVARD,LUIS DONALDO COLOSIO,2351.0,NaN,NaN,PLANTA BAJA,NaN,NaN,ZONA COMERCIAL,GALERIAS,CENTRO Y PLAZA COMERCIAL,CENTRO COMERCIAL GALERIAS,NaN,20120,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,229,2,NaN,ATENCIONALCLIENTE@MASVISION.MXQ,WWW.MASVISION.MX,Fijo,21.923118,-102.293833,2020-11
2,8514602,+ VISION OPTICA,GVMV SA DE CV,464121,Comercio al por menor de lentes,0 a 5 personas,AVENIDA,AGUASCALIENTES SUR,PROLONGACION,PASEO DE LA ASUNCION,CALLE,GERTRUDIS BOCANEGRA,CALLE,FRAY JUNÍPERO SERRA,NaN,SN,NaN,NaN,13.0,NaN,FRACCIONAMIENTO,VILLA JARDIN,NaN,NaN,NaN,20235,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,869,42,NaN,NaN,NaN,Fijo,21.859418,-102.298205,2019-11
3,6282059,"044AGUASCALIENTES,CC ALTARIA",SERVICIOS SHASA S DE RL DE CV,463211,"Comercio al por menor de ropa, excepto de bebé...",11 a 30 personas,CALLE,NINGUNO,CALLE,NINGUNO,CALLE,NINGUNO,CALLE,NINGUNO,849.0,NaN,NaN,NaN,NaN,NaN,COLONIA,TROJES DE ALONSO,CENTRO Y PLAZA COMERCIAL,ALTARIA,1023 Y 1024,20116,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,1320,66,NaN,NaN,WWW.SHASA.COM,Fijo,21.925098,-102.290276,2010-07
4,7043395,101 COPIAS Y PAPELERIA,NaN,465311,Comercio al por menor de artículos de papelería,0 a 5 personas,CALLE,MELCHOR OCAMPO,CALLE,BENITO JUAREZ,CALLE,INDEPENDENCIA,CALLE,BENITO JUÁREZ,115.0,NaN,NaN,NaN,0.0,NaN,BARRIO,EL SALTO,NaN,NaN,NaN,20330,1,AGUASCALIENTES,10,El Llano,1,Palo Alto,143,1,NaN,NaN,NaN,Fijo,21.920380,-101.963972,2019-11


ZIP: denue_00_46591-46911_1120_csv.zip
CSV: conjunto_de_datos/denue_inegi_46591-46911_.csv
Número de columnas: 41


,id,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,nom_v_e_1,tipo_v_e_2,nom_v_e_2,tipo_v_e_3,nom_v_e_3,numero_ext,letra_ext,edificio,edificio_e,numero_int,letra_int,tipo_asent,nomb_asent,tipoCenCom,nom_CenCom,num_local,cod_postal,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,6281167,1342 ELEKTRA DEL MILENIO AGUASCALIENTES 2 ASUN...,NUEVA ELEKTRA DEL MILENIO SA DE CV,466112,Comercio al por menor de electrodomésticos men...,6 a 10 personas,AVENIDA,CC. VILLASUNCION,CALLE,VALENTE QUINTANA,CALLE,ABRAHAM GONZÁLEZ,BOULEVARD,JOSE MARIA CHAVEZ,0,SN,NaN,NaN,NaN,NaN,HACIENDA,INFONAVIT PILAR BLANCO,NaN,NaN,NaN,20289,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,1458,27,NaN,NaN,WWW.ELEKTRA.COM.MX,Fijo,21.850052,-102.293881,2010-07
1,6281352,1504 ELEKTRA DEL MILENIO,NUEVA ELEKTRA DEL MILENIO SA DE CV,466112,Comercio al por menor de electrodomésticos men...,0 a 5 personas,PRIVADA,ALLENDE,CALLE,5 DE MAYO,CALLE,BENITO JUAREZ,AVENIDA,FRANCISCO I MADERO,117,NaN,NaN,NaN,NaN,NaN,COLONIA,AGUASCALIENTES CENTRO,NaN,NaN,NaN,20000,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,638,13,NaN,NaN,WWW.ELEKTRA.COM.MX,Fijo,21.882179,-102.296906,2010-07
2,8853,2007 ALUMINIO,NaN,467114,Comercio al por menor de vidrios y espejos,0 a 5 personas,AVENIDA,MAHATMA GANDHI,CALLE,ÁNGEL GONZALEZ,AVENIDA,AGUASCALIENTES SUR,BOULEVARD,JOSÉ MARÍA CHÁVEZ,104,NaN,NaN,NaN,NaN,NaN,FRACCIONAMIENTO,PARQUE URBANO HEROES MEXICANOS,NaN,NaN,NaN,20280,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,1458,46,4.499715e+09,NaN,NaN,Fijo,21.857334,-102.294679,2010-07
3,6281967,2212 AGUASCALIENTES RINCON DE ROMOS,NUEVA ELEKTRA DEL MILENIO SA DE CV,466112,Comercio al por menor de electrodomésticos men...,6 a 10 personas,AVENIDA,MORELOS,AVENIDA,MOTOLINÍA PONIENTE,CALLE,HEROICO COLEGIO MILITAR PONIENTE,CALLE,LIBERTAD,413,NaN,NaN,NaN,NaN,NaN,COLONIA,RINCON DE ROMOS CENTRO,NaN,NaN,NaN,20400,1,AGUASCALIENTES,7,Rincón de Romos,1,Rincón de Romos,59,31,NaN,NaN,WWW.ELEKTRA.COM.MX,Fijo,22.232272,-102.320699,2010-07
4,6281175,3010 ELEKTRA DEL MILENIO,NUEVA ELEKTRA DEL MILENIO SA DE CV,466112,Comercio al por menor de electrodomésticos men...,0 a 5 personas,AVENIDA,Convención de 1914 Poniente,AVENIDA,Fundición,AVENIDA,Pedro García Rojas,CALLE,Benjamín de la Mora,0,SN,NaN,NaN,0.0,NaN,HACIENDA,OLIVARES SANTANA,NaN,NaN,NaN,20010,1,AGUASCALIENTES,1,Aguascalientes,1,Aguascalientes,322,31,NaN,NaN,WWW.ELEKTRA.COM.MX,Fijo,21.894886,-102.311428,2010-07


### 2.3 Validación de consistencia del esquema

Se verifica que los cuatro archivos DENUE 2020 utilizados para comercio al por menor contengan la misma estructura de variables.

Esta validación es necesaria antes de integrar los archivos en una sola base, ya que la concatenación requiere que los conjuntos de datos sean estructuralmente compatibles.

In [7]:
# Tomar como referencia el esquema del primer archivo
primer_zip = list(muestras_denue.keys())[0]
columnas_referencia = list(muestras_denue[primer_zip].columns)

resumen_esquema = []

for nombre_zip, muestra in muestras_denue.items():

    mismas_columnas = list(muestra.columns) == columnas_referencia

    resumen_esquema.append({
        "archivo": nombre_zip,
        "numero_columnas": muestra.shape[1],
        "mismo_esquema": mismas_columnas
    })

resumen_esquema = pd.DataFrame(resumen_esquema)

display(resumen_esquema)

,archivo,numero_columnas,mismo_esquema
0,denue_00_46111_1120_csv.zip,41,True
1,denue_00_46112-46311_1120_csv.zip,41,True
2,denue_00_46321-46531_1120_csv.zip,41,True
3,denue_00_46591-46911_1120_csv.zip,41,True


In [8]:
columnas_clave = [
    "id",
    "codigo_act",
    "nombre_act",
    "per_ocu",
    "cve_ent",
    "entidad",
    "cve_mun",
    "municipio"
]

validacion_columnas = []

for nombre_zip, muestra in muestras_denue.items():

    registro = {"archivo": nombre_zip}

    for columna in columnas_clave:
        registro[columna] = columna in muestra.columns

    validacion_columnas.append(registro)

validacion_columnas = pd.DataFrame(validacion_columnas)

display(validacion_columnas)

,archivo,id,codigo_act,nombre_act,per_ocu,cve_ent,entidad,cve_mun,municipio
0,denue_00_46111_1120_csv.zip,True,True,True,True,True,True,True,True
1,denue_00_46112-46311_1120_csv.zip,True,True,True,True,True,True,True,True
2,denue_00_46321-46531_1120_csv.zip,True,True,True,True,True,True,True,True
3,denue_00_46591-46911_1120_csv.zip,True,True,True,True,True,True,True,True


### 2.4 Revisión del diccionario oficial de datos DENUE

Se consulta el diccionario de datos incluido en los paquetes oficiales de DENUE 2020.

El objetivo es documentar formalmente:

- nombre de cada variable;
- descripción;
- tipo de información;
- longitud o formato, cuando esté disponible;
- significado de las variables que serán utilizadas posteriormente.

La definición de las variables se tomará del propio diccionario oficial y no de supuestos derivados de los nombres de las columnas.

In [10]:
# Utilizar el primer ZIP como referencia para inspeccionar
# el diccionario oficial de datos

archivo_zip_ref = archivos_zip[0]

with zipfile.ZipFile(archivo_zip_ref, "r") as z:

    archivos_diccionario = [
        nombre for nombre in z.namelist()
        if nombre.startswith("diccionario_de_datos/")
        and nombre.endswith(".csv")
    ]

    print("Archivos de diccionario encontrados:")
    for nombre in archivos_diccionario:
        print(nombre)

    if len(archivos_diccionario) == 1:

        diccionario_csv = archivos_diccionario[0]

        with z.open(diccionario_csv) as archivo:
            diccionario_denue = pd.read_csv(
                archivo,
                encoding="latin-1",
                header=1
            )

        # Eliminar posibles espacios sobrantes en los encabezados
        diccionario_denue.columns = diccionario_denue.columns.str.strip()

        print("\nDimensiones del diccionario:")
        print(diccionario_denue.shape)

        print("\nColumnas del diccionario:")
        print(diccionario_denue.columns.tolist())

        display(diccionario_denue.head(20))

Archivos de diccionario encontrados:
diccionario_de_datos/denue_diccionario_de_datos.csv

Dimensiones del diccionario:
(41, 5)

Columnas del diccionario:
['Nombre del Atributo en csv', 'Nombre del Atributo en DBF', 'Tipo de dato', 'Longitud', 'Descripción']


,Nombre del Atributo en csv,Nombre del Atributo en DBF,Tipo de dato,Longitud,Descripción
0,id,id,numérico,10,"Número de identificación del DENUE, es una cl..."
1,nom_estab,nom_estab,alfanumérico,150,Es el nombre comercial o nombre exterior con e...
2,raz_social,raz_social,alfanumérico,150,Es la forma con que está legalmente constituid...
3,codigo_act,codigo_act,alfanumérico,6,La clasificación de las actividades desarrolla...
4,nombre_act,nombre_act,alfanumérico,250,Nombre del código de actividad conforme al SCI...
5,per_ocu,per_ocu,alfanumérico,20,Comprende al personal contratado directamente ...
6,tipo_vial,tipo_vial,alfanumérico,25,Es la superficie del terreno destinada para el...
7,nom_vial,nom_vial,alfanumérico,100,Es el sustantivo propio con el cual se identif...
8,tipo_v_e_1,tipo_v_e_1,alfanumérico,40,Es la superficie del terreno destinada para el...
9,nom_v_e_1,nom_v_e_1,alfanumérico,100,Es el sustantivo propio con el cual se identif...


### 2.5 Diccionario técnico de variables DENUE utilizadas en el proyecto

A partir del diccionario oficial de DENUE se seleccionan las variables necesarias para la construcción y validación de la base municipal.

La selección se limita a variables relacionadas con:

- identificación de los establecimientos;
- clasificación de la actividad económica;
- personal ocupado;
- identificación geográfica estatal y municipal.

Las variables de domicilio, contacto y ubicación puntual no son necesarias para el objetivo econométrico del proyecto.

In [11]:
# Variables DENUE relevantes para el proyecto
variables_denue_interes = [
    "id",
    "codigo_act",
    "nombre_act",
    "per_ocu",
    "cve_ent",
    "entidad",
    "cve_mun",
    "municipio"
]

diccionario_tecnico_denue = diccionario_denue[
    diccionario_denue["Nombre del Atributo en csv"]
    .isin(variables_denue_interes)
].copy()

# Ordenar de acuerdo con nuestra lista de interés
diccionario_tecnico_denue["orden"] = (
    diccionario_tecnico_denue["Nombre del Atributo en csv"]
    .map({variable: i for i, variable in enumerate(variables_denue_interes)})
)

diccionario_tecnico_denue = (
    diccionario_tecnico_denue
    .sort_values("orden")
    .drop(columns="orden")
    .reset_index(drop=True)
)

display(diccionario_tecnico_denue)

,Nombre del Atributo en csv,Nombre del Atributo en DBF,Tipo de dato,Longitud,Descripción
0,id,id,numérico,10,"Número de identificación del DENUE, es una cl..."
1,codigo_act,codigo_act,alfanumérico,6,La clasificación de las actividades desarrolla...
2,nombre_act,nombre_act,alfanumérico,250,Nombre del código de actividad conforme al SCI...
3,per_ocu,per_ocu,alfanumérico,20,Comprende al personal contratado directamente ...
4,cve_ent,cve_ent,alfanumérico,2,Clave que identifica a la entidad federativa e...
5,entidad,entidad,alfanumérico,40,Es el sustantivo propio que identifica a la en...
6,cve_mun,cve_mun,alfanumérico,3,"Clave que identifica al Municipio estadístico,..."
7,municipio,municipio,alfanumérico,100,Es el sustantivo propio que identifica al Muni...


In [12]:
variables_encontradas = set(
    diccionario_tecnico_denue["Nombre del Atributo en csv"]
)

variables_faltantes = [
    variable
    for variable in variables_denue_interes
    if variable not in variables_encontradas
]

print("Variables solicitadas:", len(variables_denue_interes))
print("Variables encontradas:", len(variables_encontradas))
print("Variables faltantes:", variables_faltantes)

Variables solicitadas: 8
Variables encontradas: 8
Variables faltantes: []


### 2.6 Validación de consistencia de los diccionarios DENUE 2020

Se verifica que los cuatro paquetes de comercio al por menor utilizados en el análisis contengan el mismo diccionario de datos.

Este control permite confirmar que las variables poseen la misma estructura y definición antes de integrar los archivos.

In [13]:
diccionarios_denue = {}

for archivo_zip in archivos_zip:

    with zipfile.ZipFile(archivo_zip, "r") as z:

        archivos_diccionario = [
            nombre for nombre in z.namelist()
            if nombre.startswith("diccionario_de_datos/")
            and nombre.endswith(".csv")
        ]

        diccionario_csv = archivos_diccionario[0]

        with z.open(diccionario_csv) as archivo:

            df_diccionario = pd.read_csv(
                archivo,
                encoding="latin-1",
                header=1
            )

        # Limpiar encabezados
        df_diccionario.columns = df_diccionario.columns.str.strip()

        diccionarios_denue[archivo_zip.name] = df_diccionario

In [14]:
# Utilizar el primer diccionario como referencia
primer_archivo = list(diccionarios_denue.keys())[0]
diccionario_referencia = diccionarios_denue[primer_archivo]

resultado_diccionarios = []

for nombre_zip, diccionario in diccionarios_denue.items():

    mismo_diccionario = diccionario.equals(diccionario_referencia)

    resultado_diccionarios.append({
        "archivo": nombre_zip,
        "filas_diccionario": diccionario.shape[0],
        "columnas_diccionario": diccionario.shape[1],
        "mismo_diccionario": mismo_diccionario
    })

resultado_diccionarios = pd.DataFrame(resultado_diccionarios)

display(resultado_diccionarios)

,archivo,filas_diccionario,columnas_diccionario,mismo_diccionario
0,denue_00_46111_1120_csv.zip,41,5,True
1,denue_00_46112-46311_1120_csv.zip,41,5,True
2,denue_00_46321-46531_1120_csv.zip,41,5,True
3,denue_00_46591-46911_1120_csv.zip,41,5,True


### 2.7 Conteo de registros por archivo DENUE 2020

Se contabiliza el número de establecimientos contenidos en cada uno de los cuatro archivos de comercio al por menor.

La lectura se realiza por bloques (`chunks`) para evitar cargar simultáneamente millones de registros en memoria.

Este procedimiento permite:

- conocer el tamaño real de cada base;
- verificar que los archivos contienen información;
- calcular el total de establecimientos comerciales considerados para 2020;
- mantener un uso controlado de memoria.

In [15]:
resumen_registros_2020 = []

for archivo_zip in archivos_zip:

    print(f"Procesando: {archivo_zip.name}")

    with zipfile.ZipFile(archivo_zip, "r") as z:

        # Identificar el archivo principal
        archivos_datos = [
            nombre for nombre in z.namelist()
            if nombre.startswith("conjunto_de_datos/")
            and nombre.endswith(".csv")
        ]

        csv_principal = archivos_datos[0]

        total_registros = 0

        # Leer únicamente la columna id y hacerlo por bloques
        with z.open(csv_principal) as archivo_csv:

            for chunk in pd.read_csv(
                archivo_csv,
                encoding="latin-1",
                usecols=["id"],
                chunksize=200_000
            ):

                total_registros += len(chunk)

        resumen_registros_2020.append({
            "archivo": archivo_zip.name,
            "registros": total_registros
        })

        print(f"Registros encontrados: {total_registros:,}\n")

resumen_registros_2020 = pd.DataFrame(resumen_registros_2020)

display(resumen_registros_2020)

Procesando: denue_00_46111_1120_csv.zip
Registros encontrados: 604,880

Procesando: denue_00_46112-46311_1120_csv.zip
Registros encontrados: 553,190

Procesando: denue_00_46321-46531_1120_csv.zip
Registros encontrados: 570,540

Procesando: denue_00_46591-46911_1120_csv.zip
Registros encontrados: 512,387



,archivo,registros
0,denue_00_46111_1120_csv.zip,604880
1,denue_00_46112-46311_1120_csv.zip,553190
2,denue_00_46321-46531_1120_csv.zip,570540
3,denue_00_46591-46911_1120_csv.zip,512387


In [16]:
total_establecimientos_2020 = resumen_registros_2020["registros"].sum()

print(
    f"Total de establecimientos de comercio al por menor "
    f"considerados en DENUE 2020: {total_establecimientos_2020:,}"
)

Total de establecimientos de comercio al por menor considerados en DENUE 2020: 2,240,997


### 2.8 Validación de unicidad del identificador DENUE

El campo `id` identifica a cada establecimiento dentro de DENUE.

Antes de integrar los cuatro archivos se verifica:

- existencia de valores nulos en `id`;
- identificadores repetidos dentro de cada archivo;
- identificadores repetidos entre diferentes archivos;
- número total de identificadores únicos.

Esta validación evita contabilizar dos veces un mismo establecimiento en la construcción posterior de los indicadores municipales.

In [17]:
resumen_ids_2020 = []

# Conjunto acumulado de identificadores ya encontrados
ids_globales = set()

for archivo_zip in archivos_zip:

    print(f"Validando: {archivo_zip.name}")

    ids_archivo = set()
    ids_duplicados_internos = set()

    total_registros = 0
    total_nulos_id = 0

    with zipfile.ZipFile(archivo_zip, "r") as z:

        archivos_datos = [
            nombre for nombre in z.namelist()
            if nombre.startswith("conjunto_de_datos/")
            and nombre.endswith(".csv")
        ]

        csv_principal = archivos_datos[0]

        with z.open(csv_principal) as archivo_csv:

            for chunk in pd.read_csv(
                archivo_csv,
                encoding="latin-1",
                usecols=["id"],
                chunksize=200_000
            ):

                total_registros += len(chunk)

                # Convertir a numérico para validar correctamente
                ids = pd.to_numeric(
                    chunk["id"],
                    errors="coerce"
                )

                total_nulos_id += ids.isna().sum()

                ids_validos = ids.dropna().astype("int64")

                # IDs repetidos dentro del mismo chunk
                repetidos_chunk = set(
                    ids_validos[
                        ids_validos.duplicated(keep=False)
                    ].tolist()
                )

                # IDs que ya habían aparecido en chunks anteriores
                ids_chunk = set(ids_validos.tolist())

                repetidos_previos = ids_chunk.intersection(
                    ids_archivo
                )

                ids_duplicados_internos.update(
                    repetidos_chunk
                )

                ids_duplicados_internos.update(
                    repetidos_previos
                )

                ids_archivo.update(ids_chunk)

    # Revisar si existen IDs también encontrados en otros archivos
    ids_repetidos_entre_archivos = (
        ids_archivo.intersection(ids_globales)
    )

    resumen_ids_2020.append({
        "archivo": archivo_zip.name,
        "registros": total_registros,
        "id_nulos": total_nulos_id,
        "id_unicos_archivo": len(ids_archivo),
        "id_duplicados_internos": len(ids_duplicados_internos),
        "id_repetidos_entre_archivos": len(
            ids_repetidos_entre_archivos
        )
    })

    ids_globales.update(ids_archivo)

    print(f"IDs únicos: {len(ids_archivo):,}")
    print(
        f"IDs duplicados internos: "
        f"{len(ids_duplicados_internos):,}"
    )
    print(
        f"IDs compartidos con otros archivos: "
        f"{len(ids_repetidos_entre_archivos):,}"
    )
    print()

resumen_ids_2020 = pd.DataFrame(resumen_ids_2020)

display(resumen_ids_2020)

Validando: denue_00_46111_1120_csv.zip
IDs únicos: 604,880
IDs duplicados internos: 0
IDs compartidos con otros archivos: 0

Validando: denue_00_46112-46311_1120_csv.zip
IDs únicos: 553,190
IDs duplicados internos: 0
IDs compartidos con otros archivos: 0

Validando: denue_00_46321-46531_1120_csv.zip
IDs únicos: 570,540
IDs duplicados internos: 0
IDs compartidos con otros archivos: 0

Validando: denue_00_46591-46911_1120_csv.zip
IDs únicos: 512,387
IDs duplicados internos: 0
IDs compartidos con otros archivos: 0



,archivo,registros,id_nulos,id_unicos_archivo,id_duplicados_internos,id_repetidos_entre_archivos
0,denue_00_46111_1120_csv.zip,604880,0,604880,0,0
1,denue_00_46112-46311_1120_csv.zip,553190,0,553190,0,0
2,denue_00_46321-46531_1120_csv.zip,570540,0,570540,0,0
3,denue_00_46591-46911_1120_csv.zip,512387,0,512387,0,0


In [18]:
print(
    f"Registros totales observados: "
    f"{resumen_ids_2020['registros'].sum():,}"
)

print(
    f"Identificadores únicos globales: "
    f"{len(ids_globales):,}"
)

print(
    f"Valores nulos en id: "
    f"{resumen_ids_2020['id_nulos'].sum():,}"
)

Registros totales observados: 2,240,997
Identificadores únicos globales: 2,240,997
Valores nulos en id: 0


### 2.9 Validación de claves geográficas y cobertura municipal

Antes de agregar los establecimientos a nivel municipal, se valida la calidad de las claves geográficas de DENUE 2020.

Se verifican:

- valores faltantes en `cve_ent` y `cve_mun`;
- formato de las claves;
- construcción de la clave municipal `CVEGEO` de cinco posiciones;
- número de municipios presentes en cada archivo;
- número total de municipios cubiertos por los cuatro bloques comerciales;
- consistencia entre la clave municipal y los nombres de entidad y municipio.

Esta validación es necesaria porque `CVEGEO` será la llave utilizada posteriormente para integrar DENUE con las demás fuentes públicas.

In [19]:
resumen_geografico_2020 = []
catalogos_municipales = []

for archivo_zip in archivos_zip:

    print(f"Validando: {archivo_zip.name}")

    total_registros = 0
    nulos_ent = 0
    nulos_mun = 0
    claves_invalidas = 0

    municipios_archivo = set()

    with zipfile.ZipFile(archivo_zip, "r") as z:

        archivos_datos = [
            nombre for nombre in z.namelist()
            if nombre.startswith("conjunto_de_datos/")
            and nombre.endswith(".csv")
        ]

        csv_principal = archivos_datos[0]

        with z.open(csv_principal) as archivo_csv:

            for chunk in pd.read_csv(
                archivo_csv,
                encoding="latin-1",
                usecols=[
                    "cve_ent",
                    "entidad",
                    "cve_mun",
                    "municipio"
                ],
                dtype="string",
                chunksize=200_000
            ):

                total_registros += len(chunk)

                # Eliminar espacios
                chunk["cve_ent"] = chunk["cve_ent"].str.strip()
                chunk["cve_mun"] = chunk["cve_mun"].str.strip()

                nulos_ent += chunk["cve_ent"].isna().sum()
                nulos_mun += chunk["cve_mun"].isna().sum()

                # Asegurar formato oficial:
                # entidad = 2 posiciones, municipio = 3 posiciones
                chunk["cve_ent"] = chunk["cve_ent"].str.zfill(2)
                chunk["cve_mun"] = chunk["cve_mun"].str.zfill(3)

                chunk["CVEGEO"] = (
                    chunk["cve_ent"] + chunk["cve_mun"]
                )

                # Validar que CVEGEO tenga exactamente 5 dígitos
                valida = (
                    chunk["CVEGEO"].str.len().eq(5)
                    & chunk["CVEGEO"].str.isnumeric()
                )

                claves_invalidas += (~valida.fillna(False)).sum()

                municipios_validos = set(
                    chunk.loc[valida, "CVEGEO"].dropna()
                )

                municipios_archivo.update(municipios_validos)

                # Catálogo mínimo para validar nombres
                catalogo_chunk = (
                    chunk.loc[
                        valida,
                        ["CVEGEO", "entidad", "municipio"]
                    ]
                    .drop_duplicates()
                )

                catalogos_municipales.append(catalogo_chunk)

    resumen_geografico_2020.append({
        "archivo": archivo_zip.name,
        "registros": total_registros,
        "nulos_cve_ent": nulos_ent,
        "nulos_cve_mun": nulos_mun,
        "claves_invalidas": claves_invalidas,
        "municipios_unicos": len(municipios_archivo)
    })

    print(f"Municipios únicos: {len(municipios_archivo):,}")
    print(f"Nulos cve_ent: {nulos_ent:,}")
    print(f"Nulos cve_mun: {nulos_mun:,}")
    print(f"Claves inválidas: {claves_invalidas:,}")
    print()

resumen_geografico_2020 = pd.DataFrame(
    resumen_geografico_2020
)

display(resumen_geografico_2020)

Validando: denue_00_46111_1120_csv.zip
Municipios únicos: 2,465
Nulos cve_ent: 0
Nulos cve_mun: 0
Claves inválidas: 0

Validando: denue_00_46112-46311_1120_csv.zip
Municipios únicos: 2,405
Nulos cve_ent: 0
Nulos cve_mun: 0
Claves inválidas: 0

Validando: denue_00_46321-46531_1120_csv.zip
Municipios únicos: 2,368
Nulos cve_ent: 0
Nulos cve_mun: 0
Claves inválidas: 0

Validando: denue_00_46591-46911_1120_csv.zip
Municipios únicos: 2,306
Nulos cve_ent: 0
Nulos cve_mun: 0
Claves inválidas: 0



,archivo,registros,nulos_cve_ent,nulos_cve_mun,claves_invalidas,municipios_unicos
0,denue_00_46111_1120_csv.zip,604880,0,0,0,2465
1,denue_00_46112-46311_1120_csv.zip,553190,0,0,0,2405
2,denue_00_46321-46531_1120_csv.zip,570540,0,0,0,2368
3,denue_00_46591-46911_1120_csv.zip,512387,0,0,0,2306


In [20]:
#####Comprobar claves

catalogo_municipal_2020 = (
    pd.concat(
        catalogos_municipales,
        ignore_index=True
    )
    .drop_duplicates()
)

municipios_unicos_2020 = (
    catalogo_municipal_2020["CVEGEO"]
    .nunique()
)

print(
    f"Municipios únicos cubiertos por DENUE retail 2020: "
    f"{municipios_unicos_2020:,}"
)

Municipios únicos cubiertos por DENUE retail 2020: 2,465


### 2.10 Validación de consistencia entre CVEGEO y municipio

Se verifica que cada clave geográfica municipal (`CVEGEO`) esté asociada de manera consistente con una sola entidad federativa y un solo nombre de municipio.

Este control evita problemas posteriores durante la integración de DENUE con Censo, CONAPO, ILMM y CONEVAL.

In [21]:
# Número de nombres distintos asociados a cada CVEGEO
consistencia_cvegeo = (
    catalogo_municipal_2020
    .groupby("CVEGEO")
    .agg(
        entidades_distintas=("entidad", "nunique"),
        municipios_distintos=("municipio", "nunique")
    )
    .reset_index()
)

# Detectar claves con inconsistencias
cvegeo_inconsistentes = consistencia_cvegeo[
    (consistencia_cvegeo["entidades_distintas"] > 1)
    | (consistencia_cvegeo["municipios_distintos"] > 1)
]

print(
    f"CVEGEO con inconsistencias de nombre: "
    f"{len(cvegeo_inconsistentes):,}"
)

display(cvegeo_inconsistentes.head(20))

CVEGEO con inconsistencias de nombre: 0


,CVEGEO,entidades_distintas,municipios_distintos


### 2.11 Conclusión de Data Understanding — DENUE 2020

La revisión de los cuatro paquetes DENUE 2020 correspondientes al comercio al por menor permitió confirmar la calidad estructural y geográfica de la información utilizada.

Principales resultados:

- Se identificaron correctamente los cuatro archivos esperados.
- Los cuatro archivos contienen 41 variables y presentan el mismo esquema.
- Los diccionarios de datos son consistentes entre los cuatro paquetes.
- Las ocho variables seleccionadas para el proyecto están disponibles.
- Se identificaron 2,240,997 registros de establecimientos.
- Los 2,240,997 identificadores `id` son únicos.
- No se encontraron valores nulos en `id`.
- No existen identificadores duplicados dentro de los archivos ni entre los cuatro paquetes.
- No se encontraron valores faltantes en `cve_ent` ni `cve_mun`.
- Todas las claves municipales pueden construirse correctamente en formato `CVEGEO` de cinco posiciones.
- Los cuatro bloques comerciales cubren en conjunto 2,465 municipios.
- No se detectaron inconsistencias entre `CVEGEO` y los nombres de entidad o municipio.

Con base en estos controles, los archivos DENUE 2020 se consideran aptos para iniciar la fase de preparación de datos y construir la base agregada a nivel municipal.

# Fase 3 — Data Preparation

###3.1 Documentación de la transformación
## Construcción de la base municipal DENUE 2020

La unidad de observación original de DENUE es el establecimiento económico. Sin embargo, la unidad de análisis del proyecto econométrico es el municipio.

Por esta razón, los establecimientos de comercio al por menor se agregarán mediante la clave geográfica municipal `CVEGEO`.

La base resultante contendrá:

- una fila por municipio;
- clave municipal `CVEGEO`;
- entidad federativa;
- nombre del municipio;
- número total de establecimientos de comercio al por menor registrados en DENUE 2020.

La variable resultante se denominará:

`EST_RETAIL_2020`

En esta etapa no se calculará todavía la densidad comercial, ya que para ello se requiere integrar posteriormente la población municipal proveniente de CONAPO.

Tampoco se imputarán ceros a municipios no observados en DENUE. La cobertura municipal se conservará tal como aparece en la fuente hasta realizar la integración de todas las bases.

In [22]:
###3.2 Agregación de establecimientos por municipio
# Contenedores para almacenar agregaciones parciales
agregados_municipales = []
catalogos_nombres = []

for archivo_zip in archivos_zip:

    print(f"Procesando: {archivo_zip.name}")

    with zipfile.ZipFile(archivo_zip, "r") as z:

        # Identificar automáticamente el CSV principal
        archivos_datos = [
            nombre for nombre in z.namelist()
            if nombre.startswith("conjunto_de_datos/")
            and nombre.endswith(".csv")
        ]

        csv_principal = archivos_datos[0]

        with z.open(csv_principal) as archivo_csv:

            for chunk in pd.read_csv(
                archivo_csv,
                encoding="latin-1",
                usecols=[
                    "cve_ent",
                    "entidad",
                    "cve_mun",
                    "municipio"
                ],
                dtype="string",
                chunksize=200_000
            ):

                # Limpiar claves geográficas
                chunk["cve_ent"] = (
                    chunk["cve_ent"]
                    .str.strip()
                    .str.zfill(2)
                )

                chunk["cve_mun"] = (
                    chunk["cve_mun"]
                    .str.strip()
                    .str.zfill(3)
                )

                # Construir llave municipal
                chunk["CVEGEO"] = (
                    chunk["cve_ent"]
                    + chunk["cve_mun"]
                )

                # --------------------------------------------------
                # Conteo parcial de establecimientos por municipio
                # --------------------------------------------------

                agregado_chunk = (
                    chunk
                    .groupby("CVEGEO", as_index=False)
                    .size()
                    .rename(
                        columns={
                            "size": "EST_RETAIL_2020"
                        }
                    )
                )

                agregados_municipales.append(
                    agregado_chunk
                )

                # --------------------------------------------------
                # Catálogo entidad-municipio
                # --------------------------------------------------

                catalogo_chunk = (
                    chunk[
                        [
                            "CVEGEO",
                            "entidad",
                            "municipio"
                        ]
                    ]
                    .drop_duplicates()
                )

                catalogos_nombres.append(
                    catalogo_chunk
                )

    print("Archivo procesado correctamente.\n")

Procesando: denue_00_46111_1120_csv.zip
Archivo procesado correctamente.

Procesando: denue_00_46112-46311_1120_csv.zip
Archivo procesado correctamente.

Procesando: denue_00_46321-46531_1120_csv.zip
Archivo procesado correctamente.

Procesando: denue_00_46591-46911_1120_csv.zip
Archivo procesado correctamente.



In [23]:
###3.3 Consolidar los conteos

# Unir los conteos parciales generados por cada chunk
conteo_municipal_2020 = (
    pd.concat(
        agregados_municipales,
        ignore_index=True
    )
    .groupby(
        "CVEGEO",
        as_index=False
    )["EST_RETAIL_2020"]
    .sum()
)

# Asegurar tipo entero
conteo_municipal_2020["EST_RETAIL_2020"] = (
    conteo_municipal_2020["EST_RETAIL_2020"]
    .astype("int64")
)

display(conteo_municipal_2020.head(10))

,CVEGEO,EST_RETAIL_2020
0,01001,16577
1,01002,412
2,01003,940
3,01004,170
4,01005,1753
5,01006,821
6,01007,984
7,01008,176
8,01009,251
9,01010,164


In [24]:
#####################3.4 Construir catálogo municipal

catalogo_nombres_2020 = (
    pd.concat(
        catalogos_nombres,
        ignore_index=True
    )
    .drop_duplicates()
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Municipios en el catálogo:",
    f"{catalogo_nombres_2020['CVEGEO'].nunique():,}"
)

display(catalogo_nombres_2020.head(10))

Municipios en el catálogo: 2,465


,CVEGEO,entidad,municipio
0,01001,AGUASCALIENTES,Aguascalientes
1,01002,AGUASCALIENTES,Asientos
2,01003,AGUASCALIENTES,Calvillo
3,01004,AGUASCALIENTES,Cosío
4,01005,AGUASCALIENTES,Jesús María
5,01006,AGUASCALIENTES,Pabellón de Arteaga
6,01007,AGUASCALIENTES,Rincón de Romos
7,01008,AGUASCALIENTES,San José de Gracia
8,01009,AGUASCALIENTES,Tepezalá
9,01010,AGUASCALIENTES,El Llano


In [25]:
#################3.5 Construir la base municipal DENUE 2020

denue_2020_municipal = (
    catalogo_nombres_2020
    .merge(
        conteo_municipal_2020,
        on="CVEGEO",
        how="inner",
        validate="one_to_one"
    )
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Dimensiones de la base municipal DENUE 2020:"
)

print(denue_2020_municipal.shape)

display(denue_2020_municipal.head(10))

Dimensiones de la base municipal DENUE 2020:
(2465, 4)


,CVEGEO,entidad,municipio,EST_RETAIL_2020
0,01001,AGUASCALIENTES,Aguascalientes,16577
1,01002,AGUASCALIENTES,Asientos,412
2,01003,AGUASCALIENTES,Calvillo,940
3,01004,AGUASCALIENTES,Cosío,170
4,01005,AGUASCALIENTES,Jesús María,1753
5,01006,AGUASCALIENTES,Pabellón de Arteaga,821
6,01007,AGUASCALIENTES,Rincón de Romos,984
7,01008,AGUASCALIENTES,San José de Gracia,176
8,01009,AGUASCALIENTES,Tepezalá,251
9,01010,AGUASCALIENTES,El Llano,164


### 3.6 Validación de la base municipal DENUE 2020

Después de agregar los establecimientos se verifica que la transformación no haya generado pérdida ni duplicación de información.

Se comprueba:

- número de municipios;
- unicidad de `CVEGEO`;
- ausencia de valores faltantes;
- existencia de conteos positivos;
- conservación del total original de establecimientos.

In [26]:
print("CONTROL DE CALIDAD — DENUE MUNICIPAL 2020")
print("=" * 55)

print(
    "Número de municipios:",
    f"{len(denue_2020_municipal):,}"
)

print(
    "CVEGEO únicos:",
    f"{denue_2020_municipal['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    denue_2020_municipal["CVEGEO"].duplicated().sum()
)

print(
    "Valores nulos totales:",
    denue_2020_municipal.isna().sum().sum()
)

print(
    "Municipios con conteo <= 0:",
    (
        denue_2020_municipal["EST_RETAIL_2020"]
        <= 0
    ).sum()
)

print(
    "Total de establecimientos después de agregar:",
    f"{denue_2020_municipal['EST_RETAIL_2020'].sum():,}"
)

CONTROL DE CALIDAD — DENUE MUNICIPAL 2020
Número de municipios: 2,465
CVEGEO únicos: 2,465
CVEGEO duplicados: 0
Valores nulos totales: 0
Municipios con conteo <= 0: 0
Total de establecimientos después de agregar: 2,240,997


In [27]:
####################3.7 Validación automática con assert

assert len(denue_2020_municipal) == municipios_unicos_2020, \
    "El número de municipios no coincide."

assert (
    denue_2020_municipal["CVEGEO"].nunique()
    == municipios_unicos_2020
), "Existen problemas de unicidad en CVEGEO."

assert (
    denue_2020_municipal["CVEGEO"]
    .duplicated()
    .sum()
    == 0
), "Existen CVEGEO duplicados."

assert (
    denue_2020_municipal
    .isna()
    .sum()
    .sum()
    == 0
), "Existen valores faltantes."

assert (
    denue_2020_municipal["EST_RETAIL_2020"].sum()
    == total_establecimientos_2020
), "El total de establecimientos no coincide."

assert (
    denue_2020_municipal["EST_RETAIL_2020"]
    .gt(0)
    .all()
), "Existen municipios con conteos no positivos."

print("Todas las validaciones fueron superadas correctamente.")

Todas las validaciones fueron superadas correctamente.


3.8 Guardar la base procesada
### 3.8 Exportación de la base municipal DENUE 2020

Una vez superados los controles de calidad, la base agregada a nivel municipal se guarda como archivo procesado.

Este archivo constituye el resultado final de la preparación de DENUE 2020 y será utilizado posteriormente para integrar la información comercial con las fuentes demográficas, laborales, educativas y socioeconómicas.

La base contiene una observación por municipio y la variable `EST_RETAIL_2020`, que representa el número de establecimientos de comercio al por menor registrados en DENUE 2020.


In [28]:
# Ruta de salida
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Asegurar que exista la carpeta
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Archivo final
archivo_salida_2020 = (
    PROCESSED_DIR / "denue_2020_municipal.csv"
)

# Exportar
denue_2020_municipal.to_csv(
    archivo_salida_2020,
    index=False,
    encoding="utf-8-sig"
)

print("Base DENUE 2020 guardada correctamente.")
print(f"Ruta: {archivo_salida_2020}")

Base DENUE 2020 guardada correctamente.
Ruta: c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\denue_2020_municipal.csv


In [29]:
##################3.9 Comprobar que el archivo realmente existe
print(
    "¿Existe el archivo?:",
    archivo_salida_2020.exists()
)

print(
    "Tamaño del archivo:",
    f"{archivo_salida_2020.stat().st_size / 1024:.2f} KB"
)

¿Existe el archivo?: True
Tamaño del archivo: 94.34 KB


In [30]:
###prueba para volver a leer el archivo que acabamos de guardar.

denue_2020_verificacion = pd.read_csv(
    archivo_salida_2020,
    dtype={"CVEGEO": "string"}
)

print("Dimensiones:", denue_2020_verificacion.shape)

print(
    "Total de establecimientos:",
    f"{denue_2020_verificacion['EST_RETAIL_2020'].sum():,}"
)

display(denue_2020_verificacion.head())

Dimensiones: (2465, 4)
Total de establecimientos: 2,240,997


,CVEGEO,entidad,municipio,EST_RETAIL_2020
0,01001,AGUASCALIENTES,Aguascalientes,16577
1,01002,AGUASCALIENTES,Asientos,412
2,01003,AGUASCALIENTES,Calvillo,940
3,01004,AGUASCALIENTES,Cosío,170
4,01005,AGUASCALIENTES,Jesús María,1753
